# Trace two bids without a detailed BOQ — `N`/unconfirmed vs. `null`

Unity Catalog: `ingestion_framework_test.bid_data_exploration`

Contrasts **D-111808** — our established real construction tender, already known from `FINDINGS.md`'s headline finding to have only one lump-sum `quotationline` row per vendor, no itemized breakdown — against a genuinely **`null`-flagged** example, tracing both through every table the same way `07_trace_bid_with_boq.ipynb` does for a `Y` example.

This also finally answers the still-open question from `01_rfq.ipynb`'s Run 3: what does D-111808's own `DETAILBOQAVAILABLE` value actually say? If it turns out to be `N`, that's a clean match to what we already know from its `quotationline` data. If both D-111808 and the `null` example look the same end-to-end, that's evidence `null` really does mean "no BOQ" in practice. If the `null` example actually has rich `quotationline` data, that's evidence `null` just means "never assessed", not "no BOQ" — directly resolving the open question flagged in `FINDINGS.md`.

## Part 1 — D-111808

Already partially traced in `02_rfqvendor.ipynb` (22 invited vendors), `03_quotationline.ipynb` (8 priced lump-sum lines), and `04_altquotationline.ipynb` (0 alternates). This section pulls everything into one place, plus the two pieces never checked yet: its actual `rfq` row (including the real `DETAILBOQAVAILABLE` value) and any attached documents.

### `rfq` — header (first time checking this specific row)

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfq WHERE RFQNUM = 'D-111808'

### `rfqvendor` — invited vendors (22 expected)

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfqvendor WHERE RFQNUM = 'D-111808' ORDER BY VENDOR

### `quotationline` — priced lines (8 expected, one lump-sum row per vendor)

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.quotationline WHERE RFQNUM = 'D-111808' ORDER BY VENDOR

### `altquotationline` — alternates (0 expected)

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.altquotationline WHERE RFQNUM = 'D-111808'

### `docinfo` / `doclinks` / `vw_rfqvendor_documents` — attached documents
Never checked for this tender. Schemas unknown — start with `DESCRIBE` (same as `07_trace_bid_with_boq.ipynb` — results should match since it's the same tables).

In [ ]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.docinfo

In [ ]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.doclinks

In [ ]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.vw_rfqvendor_documents

In [ ]:
%sql
-- SELECT * FROM ingestion_framework_test.bid_data_exploration.vw_rfqvendor_documents WHERE RFQNUM = 'D-111808'  -- fix column name once DESCRIBE above shows real columns

## Part 2 — a genuinely `null`-flagged example
Find a `null`-flagged RFQ that actually has substantial `quotationline` data, so the comparison to Part 1 and to `07_trace_bid_with_boq.ipynb`'s `Y` example is fair (not just picking an obscure, genuinely-empty one).

In [ ]:
%sql
SELECT r.RFQNUM, r.DESCRIPTION, r.ORGID, r.ENTERDATE,
       COUNT(ql.QUOTATIONLINEID) AS line_count,
       COUNT(DISTINCT ql.VENDOR) AS vendor_count,
       COUNT(DISTINCT ql.BOQITEMNUM) AS distinct_boqitems
FROM ingestion_framework_test.bid_data_exploration.rfq r
LEFT JOIN ingestion_framework_test.bid_data_exploration.quotationline ql ON r.RFQNUM = ql.RFQNUM
WHERE r.DETAILBOQAVAILABLE IS NULL
GROUP BY r.RFQNUM, r.DESCRIPTION, r.ORGID, r.ENTERDATE
ORDER BY line_count DESC
LIMIT 20

Set the RFQNUM to trace below once you've picked one from the candidates above.

In [ ]:
dbutils.widgets.text("null_rfqnum", "", "null-flagged RFQNUM to trace")

### `rfq` — header

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfq WHERE RFQNUM = :null_rfqnum

### `rfqvendor`

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfqvendor WHERE RFQNUM = :null_rfqnum ORDER BY VENDOR

### `quotationline`

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.quotationline WHERE RFQNUM = :null_rfqnum ORDER BY VENDOR, BOQITEMNUM, RFQLINENUM

### `altquotationline`

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.altquotationline WHERE RFQNUM = :null_rfqnum ORDER BY VENDOR, RFQLINENUM

### documents
Reuse the schemas discovered in Part 1 (same tables) — just filter:

In [ ]:
%sql
-- SELECT * FROM ingestion_framework_test.bid_data_exploration.vw_rfqvendor_documents WHERE RFQNUM = :null_rfqnum  -- fix column name once Part 1's DESCRIBE shows real columns

## Observations — fill in once both parts have real results

| | D-111808 (`N`/unconfirmed) | `null` example | `Y` example (`07`) |
|---|---|---|---|
| `rfq.DETAILBOQAVAILABLE` actual value | | | Y |
| `quotationline` row count | 8 (known) | | |
| Itemized (`BOQITEMNUM` populated, multiple per vendor)? | No (known) | | |
| Documents actually attached? | | | |

_(fill in blanks once run — this table is the real answer to "does `DETAILBOQAVAILABLE` mean what we think")_